# Урок 9. Метрики классификации
**Библиотеки:** numpy, sklearn, matplotlib

**Цель:** научиться видеть, когда accuracy врёт, и читать confusion matrix.

## Простыми словами
**Accuracy** — доля правильных ответов. Звучит честно, но: если больных 1 из 100,
модель «все здоровы» имеет accuracy 99% и при этом бесполезна. 

**Confusion matrix** — таблица: что было vs что сказала модель.
- TP: больной, нашли ✔  | FN: больной, ПРОПУСТИЛИ ✘ (самое страшное)
- FP: здоровый, ложная тревога | TN: здоровый, верно

**Precision** = из всех «тревог» — сколько настоящих. **Recall** = из всех больных — скольких нашли.
**F1** = баланс precision и recall. Между precision и recall всегда компромисс.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt

# Несбалансированные данные: 95% здоровых, 5% больных
rng = np.random.default_rng(3)
n = 1000
sick = (rng.uniform(0, 1, n) < 0.05).astype(int)
feature = sick * rng.normal(7, 2, n) + (1 - sick) * rng.normal(3, 2, n)  # анализ крови
X = feature.reshape(-1, 1)

X_tr, X_te, y_tr, y_te = train_test_split(X, sick, test_size=0.3, random_state=42, stratify=sick)

# "Глупая" модель: все здоровы
dummy_pred = np.zeros_like(y_te)
print("Глупая модель: accuracy =", accuracy_score(y_te, dummy_pred).round(3), "<- выглядит отлично!")
print("Но больных она нашла:", dummy_pred[y_te == 1].sum(), "из", y_te.sum())

clf = LogisticRegression().fit(X_tr, y_tr)
y_pred = clf.predict(X_te)
print("\nНастоящая модель:")
print(classification_report(y_te, y_pred, target_names=['здоров', 'болен']))

In [ ]:
cm = confusion_matrix(y_te, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['здоров', 'болен']).plot(cmap='Blues')
plt.title('Confusion matrix'); plt.show()
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")
print(f"precision = TP/(TP+FP) = {tp/(tp+fp):.2f}")
print(f"recall    = TP/(TP+FN) = {tp/(tp+fn):.2f}")

## Что произошло
«Всегда здоров» дала accuracy ~95%, но recall = 0 — НИ ОДНОГО больного. Accuracy обманула.
Classification report показывает precision/recall/F1 по каждому классу — смотри на редкий класс!
stratify=sick в split сохраняет пропорцию классов в train и test.

## Типичные ошибки
1. Смотреть только на accuracy при несбалансированных классах.
2. Путать precision и recall. Мнемоника: recall — «всех ли вспомнили», precision — «точны ли тревоги».
3. Оптимизировать одну метрику, забыв о цене ошибок в реальности (FN в медицине ≠ FN в спаме).

## Конспект 📓
Confusion matrix: TP/TN/FP/FN. precision=TP/(TP+FP), recall=TP/(TP+FN), F1 — их баланс.
Accuracy врёт на несбалансированных данных. classification_report — главный отчёт. stratify при split.

---
## Мини-задание 💪
Снизь порог классификации до 0.2 (через predict_proba). Пересчитай precision и recall. Что выросло, что упало? Почему это компромисс?

## 5 проверочных вопросов ❓
1. Когда accuracy обманывает? Пример.
2. Расшифруй TP, FP, FN, TN для задачи «спам-фильтр».
3. Precision vs recall — разница своими словами.
4. Для какой задачи важнее recall, а для какой precision? По примеру.
5. Что такое F1 и зачем оно?

## Домашнее задание 🏠
Задачи 17–18, 27–28, 45 из practice_tasks.md.

> Не понял что-то? Открой prompts_for_future.md — промпты 1–6 помогут разобрать любую тему с AI.
